# IMDB Sentiment Analysis

In [1]:
%pip -q install datasets

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpcore 1.0.6 requires h11<0.15,>=0.13, but you have h11 0.16.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\habib\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load a balanced IMDB subset

In [14]:
from datasets import load_dataset
from collections import Counter, defaultdict
from html import unescape
import re
import random

SEED = 89

raw = load_dataset("stanfordnlp/imdb")

def balanced_split(split, n_per_class, seed=SEED):
    rows = list(split)
    random.Random(seed).shuffle(rows)
    negative_reviews = []
    positive_reviews = []

    for row in rows:
        if row["label"] == 0 and len(negative_reviews) < n_per_class:
            negative_reviews.append(row)
        elif row["label"] == 1 and len(positive_reviews) < n_per_class:
            positive_reviews.append(row)

        if len(negative_reviews) == n_per_class and len(positive_reviews) == n_per_class:
            break

    if len(negative_reviews) < n_per_class or len(positive_reviews) < n_per_class:
        raise ValueError("The dataset split does not contain enough examples for each label.")

    selected = negative_reviews + positive_reviews
    random.Random(seed).shuffle(selected)
    return selected

train_data = balanced_split(raw["train"], 2500)
test_data = balanced_split(raw["test"], 500)

print(f"Training reviews: {len(train_data)} | labels: {dict(Counter(x['label'] for x in train_data))}")
print(f"Test reviews:     {len(test_data)} | labels: {dict(Counter(x['label'] for x in test_data))}")
print(f"Raw review example, Label = {train_data[0]['label']}:\n", train_data[0]["text"][:700])


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training reviews: 5000 | labels: {0: 2500, 1: 2500}
Test reviews:     1000 | labels: {1: 500, 0: 500}
Raw review example, Label = 0:
 I don't believe there has ever been a more evil or wicked television program to air in the United States as The 700 Club. They are today's equivalent to the Ku Klux Klan of the 20th century. Their hatred of all that is good and sweet and human and pure is beyond all ability to understand. Their daily constant attacks upon millions and millions of Americans, as well as billions of humans the world over, who don't happen to share their bigoted, cruel, monstrous, and utterly insane view of humanity is beyond anything television has ever seen. The lies they spout and the ridiculous lies they try to pass off as truth, such as the idea of "life after death" or "god" or "sin" or "the devil" is so p


## 2. Pre-processing and tokenization

In [16]:
START_TOKEN = "<s>"
END_TOKEN = "</s>"
UNK_TOKEN = "<unk>"

def preprocess_text(text):
    text = unescape(text)   
    text = re.sub(r"<[^>]+>", " ", text)        
    text = text.lower()
    text = re.sub(r"[^a-z0-9']+", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    tokens = preprocess_text(text).split()
    return [START_TOKEN] + tokens + [END_TOKEN]

example_raw = train_data[0]["text"]
print("Cleaned text example:\n", preprocess_text(example_raw)[:500])
print("Tokens example:\n", tokenize(example_raw)[:35])


Cleaned text example:
 i don't believe there has ever been a more evil or wicked television program to air in the united states as the 700 club they are today's equivalent to the ku klux klan of the 20th century their hatred of all that is good and sweet and human and pure is beyond all ability to understand their daily constant attacks upon millions and millions of americans as well as billions of humans the world over who don't happen to share their bigoted cruel monstrous and utterly insane view of humanity is beyo
Tokens example:
 ['<s>', 'i', "don't", 'believe', 'there', 'has', 'ever', 'been', 'a', 'more', 'evil', 'or', 'wicked', 'television', 'program', 'to', 'air', 'in', 'the', 'united', 'states', 'as', 'the', '700', 'club', 'they', 'are', "today's", 'equivalent', 'to', 'the', 'ku', 'klux', 'klan', 'of']


## 3. Build positive and negative bigram language models

In [17]:
train_tokens = [tokenize(row["text"]) for row in train_data]
word_vocabulary = {token for sequence in train_tokens for token in sequence}
word_vocabulary.add(UNK_TOKEN)
VOCAB_SIZE = len(word_vocabulary)

def map_unk(tokens, vocabulary=word_vocabulary):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]

def count_bigrams(token_sequences):
    bigram_counts = Counter()
    context_counts = Counter()
    for tokens in token_sequences:
        for first, second in zip(tokens, tokens[1:]):
            bigram_counts[(first, second)] += 1
            context_counts[first] += 1
    return bigram_counts, context_counts

def fit_bigram_lm(token_sequences):
    bigrams, contexts = count_bigrams(token_sequences)
    return {"bigram_counts": bigrams, "context_counts": contexts, "vocab_size": VOCAB_SIZE}

positive_sequences = [map_unk(tokens) for row, tokens in zip(train_data, train_tokens) if row["label"] == 1]
negative_sequences = [map_unk(tokens) for row, tokens in zip(train_data, train_tokens) if row["label"] == 0]

positive_lm = fit_bigram_lm(positive_sequences)
negative_lm = fit_bigram_lm(negative_sequences)

print(f"Vocabulary size: {VOCAB_SIZE:,}")
print(f"Positive bigram count: {sum(positive_lm['bigram_counts'].values()):,}")
print(f"Negative bigram count: {sum(negative_lm['bigram_counts'].values()):,}")

Vocabulary size: 42,224
Positive bigram count: 596,222
Negative bigram count: 572,468


## 4. Log-likelihood and perplexity

In [57]:
import math

def log_probability(lm, first, second):
    num = lm["bigram_counts"][(first, second)] + 1
    denum = lm["context_counts"][first] + lm["vocab_size"]
    return math.log(num / denum)

def score_document(text, lm):
    tokens = map_unk(tokenize(text))
    log_likelihood = sum(
        log_probability(lm, first, second)
        for first, second in zip(tokens, tokens[1:])
    )
    number_of_bigrams = len(tokens) - 1
    perplexity = math.exp(-log_likelihood / number_of_bigrams)
    return log_likelihood, perplexity

def compare_lms(text):
    pos_ll, pos_ppl = score_document(text, positive_lm)
    neg_ll, neg_ppl = score_document(text, negative_lm)
    return {
        "positive_log_likelihood": pos_ll,
        "positive_perplexity": pos_ppl,
        "negative_log_likelihood": neg_ll,
        "negative_perplexity": neg_ppl,
    }

example_result = compare_lms(test_data[0]["text"])
print("True label:", "positive" if test_data[0]["label"] == 1 else "negative")
for metric, value in example_result.items():
    print(f"{metric}: {value:.4f}")

True label: positive
positive_log_likelihood: -1227.0109
positive_perplexity: 4732.1939
negative_log_likelihood: -1254.7104
negative_perplexity: 5728.3053


# Part 2: Naive Bayes


### 1. Build the Vocabulary 

In [ ]:
from collections import Counter, defaultdict
def extract_features(tokens):
    # vocab of unigram and bigrams
    bigrams=[]
    for first,second in zip(tokens,tokens[1:]):
        bigram=" ".join((first,second))
        bigrams.append(bigram)
    return tokens+bigrams

def build_vocab(train_tokens,vocab_size=2000):
    feature_counts=Counter()
    feature_index_dict={}
    for review_tokens in train_tokens:
        d=Counter(extract_features(review_tokens))
        feature_counts.update(d)
    top_features=feature_counts.most_common(vocab_size)
    
    for i,(feature,_) in enumerate(top_features):
        feature_index_dict[feature]=i
    return feature_index_dict
feature_to_index=build_vocab(train_tokens)




### 2. Class-condtional Probabilities

Count of Occurences for each feature in both positive and negative reviews

In [48]:
import numpy as np

def count_features(token_sequences,feature_index_dict):
    counts=np.zeros(len(feature_index_dict))
    for token in token_sequences:
        for feature in extract_features(token):
            if feature in feature_index_dict:
                counts[feature_index_dict[feature]]+=1
    return counts
    
positive_tokens = [tokens for row, tokens in zip(train_data, train_tokens) if row["label"] == 1]
negative_tokens = [tokens for row, tokens in zip(train_data, train_tokens) if row["label"] == 0]

positive_counts = count_features(positive_tokens, feature_to_index)
negative_counts = count_features(negative_tokens, feature_to_index)

print(f"Positive total feature occurrences: {int(positive_counts.sum()):,}")
print(f"Negative total feature occurrences: {int(negative_counts.sum()):,}")
print()

# Top 5 features per class
index_to_feature = {index: feature for feature, index in feature_to_index.items()}
print(f"{'Feature':<20}{'Positive':>10}{'Negative':>10}")
print("-" * 40)
for index in range(5):
    feature = index_to_feature[index]
    print(f"{feature:<20}{int(positive_counts[index]):>10,}{int(negative_counts[index]):>10,}")


Positive total feature occurrences: 581,651
Negative total feature occurrences: 574,798

Feature               Positive  Negative
----------------------------------------
the                     34,557    32,010
and                     18,122    14,602
a                       16,836    15,657
of                      15,638    13,612
to                      13,244    13,499


Laplace Smoothed probabilities

In [50]:
def smoothed_probs(counts):
    v=len(counts)
    n=counts.sum()
    probs=np.zeros(v)
    for i in range(v):
        probs[i]=(counts[i]+1)/(n+ v)
    return probs

positive_probs = smoothed_probs(positive_counts)
negative_probs = smoothed_probs(negative_counts)

print(np.isclose(positive_probs.sum(),1.0))
print(np.isclose(negative_probs.sum(),1.0))


True
True


### 3. Log-ratios(r)

In [65]:
def log_ratios(p1,p2):
    return np.log(p1) - np.log(p2)

def log_prior_ratio(p1,p2):
    n_p1=len(p1)
    n_p2=len(p2)
    return math.log(n_p1/n_p2)
    
r=log_ratios(positive_probs,negative_probs)

### 4. Naive Bayes Prediction

In [66]:
def predict_naive_bayes(text,feature_to_index,r,prior):
    features=extract_features(tokenize(text))
    score=prior
    for feature in features:
        if feature in feature_to_index:
            score+=r[feature_to_index[feature]]
    label=1 if score>0 else 0
    return label,score

prior = log_prior_ratio(positive_tokens, negative_tokens) 

label, score = predict_naive_bayes(test_data[0]["text"], feature_to_index, r, prior)

print(f"Test Example:{test_data[0]['text']}")
print(f"True Label: {test_data[0]['label']}")
print(f"Predicted: {label}  (score {score:.3f})  ")

Test Example:NIGHTS IN RODANTHE brings back to the screen two talented actors in Diane Lane and Richard Gere in a simply beautiful story of a man and a woman hungry for something more in their lives than they have at present. The chemistry between Lane and Gere is magical from the first scene in the film to their last embrace. The locations, beauty of their attraction for one another when it unfolds when they first meet, and the story that follows, and as they begin to know each other with the attraction they feel towards each other is real, is romance that is projected to an audience with tender care. James Franco in another micro role is just the right casting, and the elegance of Lane in combination with the beach house, is a true Fall 2008 film to remember forever, as was THE NOTEBOOK.
True Label: 1
Predicted: 1  (score 20.051)  


Evaluation Metrics

In [67]:
def accuracy(tp, fp, fn, tn):
    return (tp + tn) / (tp + fp + fn + tn)

def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_score(prec, rec):
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def confusion_counts(predictions, true_labels):
    tp = fp = fn = tn = 0
    for predicted, actual in zip(predictions, true_labels):
        if predicted == 1 and actual == 1:
            tp += 1
        elif predicted == 1 and actual == 0:
            fp += 1
        elif predicted == 0 and actual == 1:
            fn += 1
        else:
            tn += 1
    return tp, fp, fn, tn

### 5. Evaluation over the test data

In [64]:
predictions=[]
scores=[]
true_labels=[row['label'] for row in test_data]

for row in test_data:
    label,score= predict_naive_bayes(row['text'],feature_to_index,r,prior)
    predictions.append(label)
    scores.append(score)
    
tp, fp, fn, tn = confusion_counts(predictions, true_labels)
acc=accuracy(tp,fp,fn,tn)
prec=precision(tp,fp)
rec=recall(tp,fn)
f1=f1_score(prec,rec)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.8150
Precision: 0.8195
Recall:    0.8080
F1-score:  0.8137
